# Como-Article Ground/Vehicle Full Corpus

Spec: `docs/specs/0006-como-article-ground-vehicle-mvp.md`

This notebook runs the same optimized extraction workflow as the single-shard MVP, but defaults to all local brWaC clean shards and a separate full-corpus bronze path so results can be compared without overwriting the baseline or half-corpus bronze datasets.

```text
Full-corpus bronze segments
  -> native Spark rlike prefilter for curated GROUND como article
  -> Python UDF extraction only on plausible rows
  -> gold Parquet and WebApp-ready CSV outputs
```


In [ ]:
import os
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

REPO_ROOT


In [ ]:
import os
import tempfile
import zipfile

from pyspark.sql import SparkSession

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-como-article-ground-vehicle-full-corpus")
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(256 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))

spark.sparkContext.addPyFile(str(package_zip))
SPARK_MASTER, spark.version


In [ ]:
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.como_article_ground_vehicle import (
    COMO_ARTICLE_BACKEND_EXPORT_DIR,
    COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH,
    COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    como_article_examples_dataframe,
    como_article_ground_counts_dataframe,
    como_article_ground_vehicle_counts_dataframe,
    como_article_review_sample_dataframe,
    como_article_vehicle_counts_dataframe,
    como_article_vehicle_ground_counts_dataframe,
    prefilter_como_article_ground_vehicle_bronze_dataframe,
    prepare_como_article_ground_vehicle_dataframe,
    write_como_article_backend_export,
    write_como_article_csv_outputs,
    write_como_article_ground_vehicle_parquet,
)


## Load Or Build Bronze

The extractor expects the bronze schema from notebook 02. If bronze output is absent, this notebook builds it from the configured raw shard.


In [ ]:
FULL_CORPUS_RAW_INPUT = "data/raw/brwac-clean-*.txt.gz"
FULL_CORPUS_BRONZE_PATH = "data/bronze/brwac_segments_full"

bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", FULL_CORPUS_BRONZE_PATH)
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", FULL_CORPUS_RAW_INPUT)

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path)

print(f"Raw input: {raw_path}")
print(f"Bronze path: {bronze_path}")
bronze_df.select("source_file").distinct().orderBy("source_file").show(20, truncate=False)

bronze_df.printSchema()


## Prefilter And Extract Spec-0006 Candidates

The native prefilter is intentionally broad but cheap. It reduces the number of rows sent through the Python UDF.


In [ ]:
prefiltered_bronze_df = prefilter_como_article_ground_vehicle_bronze_dataframe(bronze_df).persist()
prefiltered_count = prefiltered_bronze_df.count()
print(f"Prefiltered bronze rows: {prefiltered_count:,}")

candidates_df = prepare_como_article_ground_vehicle_dataframe(prefiltered_bronze_df).persist()
candidate_count = candidates_df.count()
print(f"Spec-0006 candidates: {candidate_count:,}")
candidates_df


In [ ]:
candidates_df.printSchema()


## Write Gold Dataset And Visualization CSVs


In [ ]:
write_como_article_ground_vehicle_parquet(candidates_df, REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH)
write_como_article_csv_outputs(
    candidates_df,
    ground_vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    vehicle_ground_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    ground_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    examples_path=REPO_ROOT / COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    review_sample_path=REPO_ROOT / COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
)
write_como_article_backend_export(
    candidates_df,
    export_dir=REPO_ROOT / COMO_ARTICLE_BACKEND_EXPORT_DIR,
)


## Inspect Visualization Tables

These displays use small materialized pandas tables for readability. The durable CSV outputs above remain the source of truth for downstream visualization work.


In [ ]:
from pyspark.sql.functions import col

def display_table(spark_df, limit=30):
    display(spark_df.limit(limit).toPandas())



### Backend Export Outputs


In [ ]:
import json

backend_export_dir = REPO_ROOT / COMO_ARTICLE_BACKEND_EXPORT_DIR
backend_manifest_path = backend_export_dir / "manifest.json"
backend_manifest = json.loads(backend_manifest_path.read_text(encoding="utf-8"))

print(f"Backend export: {backend_export_dir}")
print(f"Backend manifest: {backend_manifest_path}")
backend_manifest


### Ground -> Vehicle Pairs


In [ ]:
display_table(
    como_article_ground_vehicle_counts_dataframe(candidates_df).select(
        col("ground_lemma").alias("ground"),
        col("vehicle_head_lemma").alias("vehicle_head"),
        "count",
    ),
    limit=40,
)


### Vehicle -> Ground Pairs


In [ ]:
display_table(
    como_article_vehicle_ground_counts_dataframe(candidates_df).select(
        col("vehicle_head_lemma").alias("vehicle_head"),
        col("ground_lemma").alias("ground"),
        "count",
    ),
    limit=40,
)


### Ground Counts


In [ ]:
display_table(como_article_ground_counts_dataframe(candidates_df), limit=40)


### Vehicle Counts


In [ ]:
display_table(como_article_vehicle_counts_dataframe(candidates_df), limit=40)


### Examples


In [ ]:
display_table(
    como_article_examples_dataframe(candidates_df, limit=100).select(
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "candidate_full_text",
    ),
    limit=50,
)


### Review Sample


In [ ]:
display_table(
    como_article_review_sample_dataframe(candidates_df, limit=100).select(
        "ground_type",
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "confidence",
        "needs_review",
        "candidate_full_text",
    ),
    limit=50,
)
